In [1]:
#IMPORT CODE

%matplotlib widget

import os
import geopandas as gpd
import pandas as pd
import matplotlib.pyplot as plt
from cartopy.feature import ShapelyFeature
import cartopy.crs as ccrs
import matplotlib.patches as mpatches
import matplotlib.lines as mlines
import folium
import matplotlib.colors as mcolors #function for normalizing colours for colourbar features
import numpy as np
plt.ion() # make the plotting interactive

#LOAD DATA

res_route = gpd.read_file('London_data/restricted_route.json')

cycle = gpd.read_file('London_data/cycle_lane_track.json')  #2017/18 cycle routes
london = gpd.read_file('London_data/London_Boroughs.gpkg')
method_new = pd.read_csv('London_data/method.csv', skiprows=6)
method_old = pd.read_csv('London_data/method.csv.csv', skiprows=6)

In [2]:
#filter the data to show true cycle infrastructure
cycle_t = cycle.drop(columns=['CLT_CARR',
                              'CLT_SHARED',
                              'CLT_ADVIS',
                              'CLT_PRIORI',
                              'CLT_BBYPAS',
                              'CLT_PARKR',
                              'CLT_WATERR',
                              'CLT_PTIME',
                              'CLT_ACCESS',
                              'CLT_COLOUR',
                              'PHOTO1_URL',
                              'PHOTO2_URL'])


In [3]:

cycle_t = cycle_t[(cycle_t == 'TRUE').any(axis=1)]
cycle_t.head()

,FEATURE_ID,SVDATE,CLT_SEGREG,CLT_STEPP,CLT_PARSEG,CLT_MANDAT,CLT_CONTRA,CLT_BIDIRE,CLT_CBYPAS,BOROUGH,geometry
4,RWG155003,2017-10-17,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,Barking & Dagenham,"LINESTRING (0.14087 51.5591, 0.14069 51.55909,..."
19,RWG154849,2017-10-15,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,FALSE,Barking & Dagenham,"LINESTRING (0.13052 51.56912, 0.13039 51.56914)"
37,RWG155353,2017-10-20,FALSE,FALSE,FALSE,TRUE,TRUE,FALSE,FALSE,Barking & Dagenham,"LINESTRING (0.1529 51.55969, 0.1518 51.55969)"
118,RWG155561,2017-10-24,TRUE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,Barking & Dagenham,"LINESTRING (0.16346 51.52843, 0.16355 51.52853..."
119,RWG155562,2017-10-24,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,FALSE,Barking & Dagenham,"LINESTRING (0.17233 51.53665, 0.17222 51.53667..."


In [4]:
cycle.head()

,FEATURE_ID,SVDATE,CLT_CARR,CLT_SEGREG,CLT_STEPP,CLT_PARSEG,CLT_SHARED,CLT_MANDAT,CLT_ADVIS,CLT_PRIORI,...,CLT_BBYPAS,CLT_PARKR,CLT_WATERR,CLT_PTIME,CLT_ACCESS,CLT_COLOUR,BOROUGH,PHOTO1_URL,PHOTO2_URL,geometry
0,RWG154837,2017-10-14,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,...,FALSE,FALSE,FALSE,FALSE,NaN,NONE,Barking & Dagenham,https://cycleassetimages.data.tfl.gov.uk/RWG15...,https://cycleassetimages.data.tfl.gov.uk/RWG15...,"LINESTRING (0.13534 51.57164, 0.13348 51.5713)"
1,RWG154839,2017-10-14,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,...,FALSE,FALSE,FALSE,FALSE,NaN,NONE,Barking & Dagenham,https://cycleassetimages.data.tfl.gov.uk/RWG15...,https://cycleassetimages.data.tfl.gov.uk/RWG15...,"LINESTRING (0.13189 51.57102, 0.13128 51.57093..."
2,RWG154848,2017-10-14,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,TRUE,...,FALSE,FALSE,FALSE,FALSE,NaN,NONE,Barking & Dagenham,https://cycleassetimages.data.tfl.gov.uk/RWG15...,https://cycleassetimages.data.tfl.gov.uk/RWG15...,"LINESTRING (0.1353 51.56129, 0.13484 51.56133,..."
3,RWG154853,2017-10-15,TRUE,FALSE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,...,FALSE,FALSE,FALSE,FALSE,NaN,NONE,Barking & Dagenham,https://cycleassetimages.data.tfl.gov.uk/RWG15...,https://cycleassetimages.data.tfl.gov.uk/RWG15...,"LINESTRING (0.14181 51.57241, 0.1413 51.57365)"
4,RWG155003,2017-10-17,TRUE,FALSE,FALSE,FALSE,FALSE,TRUE,FALSE,FALSE,...,FALSE,FALSE,FALSE,FALSE,NaN,NONE,Barking & Dagenham,https://cycleassetimages.data.tfl.gov.uk/RWG15...,https://cycleassetimages.data.tfl.gov.uk/RWG15...,"LINESTRING (0.14087 51.5591, 0.14069 51.55909,..."


In [5]:
#delete end rows


In [6]:
#lets compare the number of people cycling to work between 2011 and 2021, and see if
#there is a pattern of cycle infrastructure promoting the use of cycles
method_old

,Area,mnemonic,All categories: Method of travel to work (2001 specification),Bicycle
0,ualad09:Barking and Dagenham,E09000002,58444.0,939.0
1,ualad09:Barnet,E09000003,129112.0,1593.0
2,ualad09:Bexley,E09000004,74670.0,927.0
3,ualad09:Brent,E09000005,115221.0,2054.0
4,ualad09:Bromley,E09000006,111405.0,1366.0
5,ualad09:Camden,E09000007,272367.0,13909.0
6,ualad09:City of London,E09000001,356706.0,13501.0
7,ualad09:Croydon,E09000008,120398.0,1524.0
8,ualad09:Ealing,E09000009,132461.0,3211.0
9,ualad09:Enfield,E09000010,105493.0,1647.0


In [7]:
#our 2011 table is missing a percentage column


In [8]:
method_old['bike%2011'] = method_old['Bicycle']/method_old['All categories: Method of travel to work (2001 specification)']*100

In [9]:
method_old.head()

,Area,mnemonic,All categories: Method of travel to work (2001 specification),Bicycle,bike%2011
0,ualad09:Barking and Dagenham,E09000002,58444.0,939.0,1.606666
1,ualad09:Barnet,E09000003,129112.0,1593.0,1.233813
2,ualad09:Bexley,E09000004,74670.0,927.0,1.241462
3,ualad09:Brent,E09000005,115221.0,2054.0,1.782661
4,ualad09:Bromley,E09000006,111405.0,1366.0,1.226157


In [10]:
method_new.head()

,Area,mnemonic,Total: All usual residents aged 16 years and over in employment the week before the census,%,"Underground, metro, light rail, tram",%.1,Train,%.2,"Bus, minibus or coach",%.3,"Motorcycle, scooter or moped",%.4,Driving a car or van,%.5,Bicycle,%.6,On foot,%.7
0,ladu2023:Barking and Dagenham,E09000002,94586.0,100.0,15301.0,16.2,8660.0,9.2,9690.0,10.2,599.0,0.6,30732.0,32.5,1218.0,1.3,4464.0,4.7
1,ladu2023:Barnet,E09000003,186175.0,100.0,20758.0,11.1,4690.0,2.5,15090.0,8.1,1197.0,0.6,46797.0,25.1,2162.0,1.2,9361.0,5.0
2,ladu2023:Bexley,E09000004,117983.0,100.0,2838.0,2.4,12164.0,10.3,7763.0,6.6,767.0,0.7,42503.0,36.0,995.0,0.8,5258.0,4.5
3,ladu2023:Brent,E09000005,160997.0,100.0,24200.0,15.0,8267.0,5.1,21439.0,13.3,1801.0,1.1,34917.0,21.7,4074.0,2.5,11000.0,6.8
4,ladu2023:Bromley,E09000006,162122.0,100.0,3146.0,1.9,15214.0,9.4,8069.0,5.0,966.0,0.6,45304.0,27.9,1784.0,1.1,7029.0,4.3


In [11]:
#find a more accurate value of percentage in our new dataset
#we could alter the significant figure length of the old dataset, 
#but we can also use the exisitng data to create a more accurate overall picture
method_new['bike_pc'] = method_new['Bicycle']/method_new['Total: All usual residents aged 16 years and over in employment the week before the census']*100

#lets add the 2021 percentages to our 2011 dataset
method_old['bike_2021'] = method_new['bike_pc']

method_old.head()

In [12]:
method_old.head()

,Area,mnemonic,All categories: Method of travel to work (2001 specification),Bicycle,bike%2011,bike_2021
0,ualad09:Barking and Dagenham,E09000002,58444.0,939.0,1.606666,1.287717
1,ualad09:Barnet,E09000003,129112.0,1593.0,1.233813,1.161273
2,ualad09:Bexley,E09000004,74670.0,927.0,1.241462,0.843342
3,ualad09:Brent,E09000005,115221.0,2054.0,1.782661,2.530482
4,ualad09:Bromley,E09000006,111405.0,1366.0,1.226157,1.100406


In [13]:


#add a column to demonstrate change in bicycle usage. will need to make a new column to avoid recounting and crashing
method_old['bike_change'] = method_old['bike_2021']-method_old['bike%2011']
method_old['bike_pc'] = method_old['bike_change']/method_old['bike_2021']*100

In [14]:
method_old

,Area,mnemonic,All categories: Method of travel to work (2001 specification),Bicycle,bike%2011,bike_2021,bike_change,bike_pc
0,ualad09:Barking and Dagenham,E09000002,58444.0,939.0,1.606666,1.287717,-0.318949,-24.768580
1,ualad09:Barnet,E09000003,129112.0,1593.0,1.233813,1.161273,-0.072540,-6.246551
2,ualad09:Bexley,E09000004,74670.0,927.0,1.241462,0.843342,-0.398121,-47.207500
3,ualad09:Brent,E09000005,115221.0,2054.0,1.782661,2.530482,0.747821,29.552505
4,ualad09:Bromley,E09000006,111405.0,1366.0,1.226157,1.100406,-0.125751,-11.427688
5,ualad09:Camden,E09000007,272367.0,13909.0,5.106713,3.895192,-1.211521,-31.102985
6,ualad09:City of London,E09000001,356706.0,13501.0,3.784910,3.568759,-0.216150,-6.056737
7,ualad09:Croydon,E09000008,120398.0,1524.0,1.265802,1.278714,0.012913,1.009808
8,ualad09:Ealing,E09000009,132461.0,3211.0,2.424110,2.871573,0.447464,15.582525
9,ualad09:Enfield,E09000010,105493.0,1647.0,1.561241,1.417996,-0.143245,-10.101932


In [15]:
methold_old = method_old.drop(range(34,36))
#this is not working for some reason


method_old

In [16]:
method_old.head()

,Area,mnemonic,All categories: Method of travel to work (2001 specification),Bicycle,bike%2011,bike_2021,bike_change,bike_pc
0,ualad09:Barking and Dagenham,E09000002,58444.0,939.0,1.606666,1.287717,-0.318949,-24.768580
1,ualad09:Barnet,E09000003,129112.0,1593.0,1.233813,1.161273,-0.072540,-6.246551
2,ualad09:Bexley,E09000004,74670.0,927.0,1.241462,0.843342,-0.398121,-47.207500
3,ualad09:Brent,E09000005,115221.0,2054.0,1.782661,2.530482,0.747821,29.552505
4,ualad09:Bromley,E09000006,111405.0,1366.0,1.226157,1.100406,-0.125751,-11.427688


In [17]:
method_old = method_old.drop(range(34,37))

method_old

In [18]:
method_old

,Area,mnemonic,All categories: Method of travel to work (2001 specification),Bicycle,bike%2011,bike_2021,bike_change,bike_pc
0,ualad09:Barking and Dagenham,E09000002,58444.0,939.0,1.606666,1.287717,-0.318949,-24.768580
1,ualad09:Barnet,E09000003,129112.0,1593.0,1.233813,1.161273,-0.072540,-6.246551
2,ualad09:Bexley,E09000004,74670.0,927.0,1.241462,0.843342,-0.398121,-47.207500
3,ualad09:Brent,E09000005,115221.0,2054.0,1.782661,2.530482,0.747821,29.552505
4,ualad09:Bromley,E09000006,111405.0,1366.0,1.226157,1.100406,-0.125751,-11.427688
5,ualad09:Camden,E09000007,272367.0,13909.0,5.106713,3.895192,-1.211521,-31.102985
6,ualad09:City of London,E09000001,356706.0,13501.0,3.784910,3.568759,-0.216150,-6.056737
7,ualad09:Croydon,E09000008,120398.0,1524.0,1.265802,1.278714,0.012913,1.009808
8,ualad09:Ealing,E09000009,132461.0,3211.0,2.424110,2.871573,0.447464,15.582525
9,ualad09:Enfield,E09000010,105493.0,1647.0,1.561241,1.417996,-0.143245,-10.101932


In [19]:
#37 is still there, drop it individually
method_old = method_old.drop(37)

In [20]:
#compare CRS

print(method_old.crs, cycle_t.crs)

AttributeError: 'DataFrame' object has no attribute 'crs'

In [ ]:
#we see an error as method_old is not recognised as geodataframes
#lets create a universal crs using our bike usage table
method_old.crs

In [ ]:
method_new

In [ ]:
#lets use our london data frame and merge

london.head()

In [ ]:
travel_map = method_old.merge(london, left_on='mnemonic', right_on='gss_code', how='right')

In [ ]:
travel_map.head()

In [ ]:
travel_map

In [ ]:
#this raises an error as our travel map is not yet a geodataframe
#find out the original london crs and use this
london.crs

#note that cartopy struggles with setting EPSG UK value as ccrs, use OSGB instead




#add a shortcut function for a colour bar
def generate_handles(edgecolor, labels, alpha, travel_map, column, cmap_name='viridis'):
    handles = []
    cmap_name = plt.get_cmap(cmap_name)
    #create a vmin and vmax within function, 
    #normalize the color values in viridis to match our values
    vmin = travel_map[column].min()
    vmax = travel_map[column].max()
    sample_values = np.linspace(vmax, vmin)
    norm = mcolors.Normalize(vmin=vmin, vmax=vmax)
    for label, val in zip(labels, sample_values): #filter through whatever column we give to handles generator
        color_decimal = norm(val)
        color = cmap_name(color_decimal) #assign a color to label by 
                                    #searching cmap (it's viridis, which is now a dataset) 
                                    #for a corresponding column
        rectangle = mpatches.Rectangle((0,0), 1, 1, facecolor=color, edgecolor=edgecolor, alpha=1, label=label)
                                    #rectangle is now 0 to 1 colurs (formerly -60 to 0)
        handles.append(rectangle)
    return handles
colourbar = generate_handles('k', ['Bike Change'], '0.8', travel_map, 'bike_change', cmap_name='viridis')
colourbar



In [ ]:
uni_crs=ccrs.OSGB()
method_new['bike_pc'] = method_new['Bicycle']/method_new['Total: All usual residents aged 16 years and over in employment the week before the census']*100
method_old['bike_2021'] = method_new['bike_pc']
method_old['bike_change'] = method_old['bike_2021']-method_old['bike%2011']
method_old['bike_pc'] = method_old['bike_change']/method_old['bike_2021']*100
method_old = method_old.drop(range(34,37))
method_old = method_old.drop(37)

In [ ]:



travel_map = method_old.merge(london, left_on='mnemonic', right_on='gss_code', how='right')
travel_map = gpd.GeoDataFrame(travel_map, geometry='geometry', crs=uni_crs)
travel_map = travel_map.to_crs(uni_crs)
#set up chloropleth style map
travel_map['coords']=travel_map.geometry.centroid


fig, ax = plt.subplots(figsize=(18,10), subplot_kw={'projection':uni_crs})


travel_map.plot(column='bike_pc',
                ax=ax,
                transform=uni_crs,
                cmap='viridis',
                legend=True,
                legend_kwds={'label':'Change in Bike Usage', 'orientation':'vertical'},
                aspect='equal')


plt.show()

travel_map.set_geometry('coords').plot(marker='^', color='lightblue', markersize=10, alpha=0.8, zorder=2, ax=ax)
#this takes a white to process. create a new column for the marker at the top of the code

#can add names for each borough to the map
london = london.to_crs(uni_crs)


text_centroids = london.centroid.translate(xoff=200, yoff=500)
#use london dataset as the names are nicer
for idx, row in london.iterrows():
    point = text_centroids[idx]
    ax.annotate(text=row['name'],
    xy = (point.x, point.y),
    horizontalalignment='right',
    fontsize=6.5,
    color='k',
               zorder=3)

#it is hard to distinguish the marker from the name label. change this by using
#translate() function in the above function

#add cycle lanes
cycle = cycle_t.to_crs(uni_crs)
cycle.plot(ax=ax, linewidth=0.3, zorder=4, color='r')


In [ ]:
help(ax.plot)

In [ ]:
print(method_old.columns)

In [ ]:
#create a function that lists the longest connecting dedicated cycle route for a given borough


#1. function that calculates length of intersecting linestrings (cycle)
#ensure there are a significant number of cycle lanes that do intersect

#create a geometric mask for a small borough for computational purposes
bexley = london.groupby('name')['geometry']


In [ ]:
print(borough_mask)

In [ ]:
bexley_clipped = gpd.clip(cycle, borough_mask['Bexley'])

In [ ]:
def borough_transit(cycle, london):
    print('Select a borough')
    user_borough = input()

    if user_borough in london['name'].values:
        sample_mask = london.loc[london['name']==user_borough, 'geometry'].iloc[0]#identifies polygon, not table
        clipped_cycle = cycle.intersection(sample_mask)
        total_length = clipped_cycle.length.sum()
        print(f'You can travel {total_length:.2f}m without leaving cycle lanes in {user_borough}')
    else:
        print('that is not a borough')
        


In [ ]:
#borough_transit(cycle, london)

In [ ]:
#the function we created is unlikely, with expected figures around the single digits
def borough_transit_rank(cycle, london, crs=uni_crs):
    print('Select a borough')
    user_borough = input()
    
    if user_borough in london['name'].values:
       
        sample_mask = london.loc[london['name']==user_borough, 'geometry'].iloc[0]#identifies polygon, not table
        #add a 1m buffer for nearby cycle tracks
        cycle_buffer = cycle_t.buffer(1)
        clipped_cycle = cycle_buffer.intersection(sample_mask)
        total_length = clipped_cycle.length.sum()
        '''create a new dataset
        sorting the intersecting tracks'''
        lengthS = clipped_cycle.length.unique()
        ranked_length = lengthS.max()
        
        print(f'You can travel {ranked_length/1000:.2f}km without leaving cycle lanes in {user_borough}')
        

        #recreate sample mask for map purposes
        london_plot = london.to_crs(epsg=4326) 
        fig1, ax1 = plt.subplots(figsize=(18,10), subplot_kw={'projection':ccrs.PlateCarree()})


        #need to recreate geometry for single polygon
        sample_mask_outline = ShapelyFeature([sample_mask], ccrs.PlateCarree(), edgecolor='k', facecolor='none', zorder=1)

        
#this takes a while to process. create a new column for the marker at the top of the code
        clipped_cycle = gpd.GeoSeries(clipped_cycle, crs=uni_crs)
        clipped_cycle = clipped_cycle.to_crs(epsg=4326)
        clipped_cycle.plot(ax=ax1, linewidth=0.3, zorder=2, color='r')
#ax1.set_extent gets confused by the large function, so place the crs call right before plot.show
        a = clipped_cycle.bounds
        ax1.set_extent([a[0], a[2], a[1], a[3]], crs=ccrs.PlateCarree())
        plt.show()
        print(sample_mask_plot.bounds)
        print(sample_mask_outline)
    else:
        print('that is not a borough')
        


In [ ]:
print(sample_mask_plot.type)

In [ ]:
borough_transit_rank(cycle, london)
#map is still plotted zoomed out. manually readjust the bounds function

In [ ]:
sample_mask = london.loc[london['name', 'geometry']]


In [ ]:
print(borough_transit_rank[maxx])

In [ ]:
print(sample_mask)

In [ ]:
print(cycle.crs)

In [ ]:
print(london.crs)

In [ ]:
print(cycle.crs==london.crs)

In [ ]:
print(type(cycle))

ax

In [ ]:
print(london.crs)


In [ ]:
new_user = input(Brent)

sample_mask = london.loc[london['name']==new_user, 'geometry'].iloc[0]#identifies polygon, not table
        #add a 1m buffer for nearby cycle tracks
cycle_buffer = cycle.buffer(1)
clipped_cycle = cycle_buffer.intersection(sample_mask)
total_length = clipped_cycle.length.sum()
'''create a new dataset
sorting the intersecting tracks'''
lengthS = clipped_cycle.length.unique()
ranked_length = lengthS.max()
        
print(f'You can travel {ranked_length/1000:.2f}km without leaving cycle lanes in {new_user}')
        

        #recreate sample mask for map purposes
london_plot = london.to_crs(epsg=4326)
sample_mask_plot = london_plot.loc[london_plot['name']==new_user, 'geometry'].iloc[0]#identifies polygon, not table
fig1, ax1 = plt.subplots(figsize=(18,10), subplot_kw={'projection':ccrs.PlateCarree()})


        #need to recreate geometry for single polygon
sample_mask_outline = ShapelyFeature(sample_mask_plot, ccrs.PlateCarree(), edgecolor='k', facecolor='none', zorder=1)
a = sample_mask_plot.bounds
        
ax1.add_feature(sample_mask_outline)
        
#this takes a while to process. create a new column for the marker at the top of the code
clipped_cyle = clipped_cycle.to_crs(epsg=4326)
clipped_cycle.plot(ax=ax1, linewidth=0.3, zorder=2, color='r')
#ax1.set_extent gets confused by the large function, so place the crs call right before plot.show
ax1.set_extent([a[0], a[2], a[1], a[3]], crs=ccrs.PlateCarree())
plt.show()
print(sample_mask_plot.bounds)

In [ ]:
london_roads = gpd.read_file('London_data/TLRN_Highway_Authority_Boundary.shp.xml')

In [ ]:
print (cycle == cycle_t)

In [ ]:
print(cycle_t['geometry']